# Week 5: RAG Pipeline with Retrieval Evaluation

Retrieval runs fully local and free; only generation needs a key (set `GEMINI_API_KEY`, else it is skipped with a notice). Dependencies: `sentence-transformers`, `faiss-cpu`. For generation: `pip install openai`.

## Part 1 Pipeline

**Corpus:** eight IETF RFCs, fetched as plain text from rfc-editor.org

In [1]:
import os, re, pathlib, urllib.request, numpy as np
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer
import faiss
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

RFCS = {8259: 'JSON', 3339: 'Date and time', 4648: 'Base16/32/64 encodings', 7519: 'JSON Web Token',
        6750: 'OAuth 2.0 bearer tokens', 6265: 'HTTP cookies', 6455: 'WebSocket', 9562: 'UUIDs'}
RFC_DIR = pathlib.Path('rfcs'); RFC_DIR.mkdir(exist_ok=True)

def load_rfc(n):
    path = RFC_DIR / f'rfc{n}.txt'
    if not path.exists():
        urllib.request.urlretrieve(f'https://www.rfc-editor.org/rfc/rfc{n}.txt', path)
    text = path.read_text(encoding='utf-8')
    # Page break = footer line, form feed, running header (absent after the last page).
    return re.sub(r'\n[^\n]*\[Page \d+\]\n\f\n(?:RFC \d+[^\n]*\n)?', '\n', text)

CORPUS = {n: load_rfc(n) for n in RFCS}
for n, text in CORPUS.items():
    print(f'RFC {n}  {RFCS[n]:24s} {len(text):7,d} chars')
print('corpus chars:', sum(map(len, CORPUS.values())))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RFC 8259  JSON                      26,065 chars
RFC 3339  Date and time             32,473 chars
RFC 4648  Base16/32/64 encodings    32,900 chars
RFC 7519  JSON Web Token            58,672 chars
RFC 6750  OAuth 2.0 bearer tokens   36,358 chars
RFC 6265  HTTP cookies              74,321 chars
RFC 6455  WebSocket                151,632 chars
RFC 9562  UUIDs                    114,636 chars
corpus chars: 527057


In [2]:
def chunk_fixed(text, size, overlap):
    text = re.sub(r'\s+', ' ', text).strip(); out, i = [], 0
    while i < len(text):
        out.append(text[i:i+size]); i += size - overlap
    return out
def chunk_paragraph(text):
    return [re.sub(r'\s+',' ',p).strip() for p in text.split('\n\n') if p.strip()]
def chunk_corpus(chunker, *args):
    # Chunk each RFC separately so no chunk spans two documents; tag each chunk with its RFC.
    return [(n, c) for n, text in CORPUS.items() for c in chunker(text, *args)]

configs = {'small (120/20)': chunk_corpus(chunk_fixed, 120, 20),
           'large (320/40)': chunk_corpus(chunk_fixed, 320, 40),
           'by-paragraph': chunk_corpus(chunk_paragraph)}
for n, c in configs.items():
    print(f'{n:16s} -> {len(c):5d} chunks')

def build(chunks):
    e = embedder.encode([t for _, t in chunks], normalize_embeddings=True).astype('float32')
    idx = faiss.IndexFlatIP(e.shape[1]); idx.add(e); return idx
def search(idx, q, k=3):
    qe = embedder.encode([q], normalize_embeddings=True).astype('float32')
    # FAISS uses -1 for missing results when fewer than k chunks exist.
    return [i for i in idx.search(qe, k)[1][0] if i >= 0]
def retrieve(idx, chunks, q, k=3):
    return [chunks[i] for i in search(idx, q, k)]

small (120/20)   ->  4653 chunks
large (320/40)   ->  1666 chunks
by-paragraph     ->  2604 chunks


## Part 2: Evaluate retrieval (compare chunking configs)
A query is answered only if the retrieved context contains the answer fact. 

In [3]:
# (query, relevant RFC, acceptable answer strings)
qa = [
    ('what text encoding must JSON use when exchanged between systems', 8259,
     ('MUST be encoded using UTF-8', 'require the use of UTF-8 when transmitted')),
    ('what character separates the date from the time in an internet timestamp', 3339,
     ('full-date "T" full-time', 'date and time separated by "T"', 'requires the "T"')),
    ('which character replaces the plus sign in URL-safe base64', 4648,
     ('62 - (minus)',)),
    ('which claim stops a JWT from being used before a certain time', 7519,
     ('the time before which the JWT MUST NOT be accepted',)),
    ('what status code should a server return for an expired access token', 6750,
     ('HTTP 401 (Unauthorized)', 'expired access token: HTTP/1.1 401')),
    ('response when the token lacks the permissions a request needs', 6750,
     ('403 (Forbidden)',)),
    ('if a cookie sets both Max-Age and Expires, which one wins', 6265,
     ('attribute has precedence',)),
    ('which opcode is used for a websocket ping frame', 6455,
     ('%x9 denotes a ping', '%x9 ; ping', '0x9 (Ping)', 'opcode of 0x9')),
    ('what protocol version does the client send in the websocket opening handshake', 6455,
     ('with a value of 13', 'MUST be 13', 'Sec-WebSocket-Version: 13')),
    ('how many bits of a UUIDv7 hold the millisecond timestamp', 9562,
     ('most significant 48 bits', '48-bit big-endian unsigned number of the Unix Epoch',
      'UUIDv7 features a 48-bit timestamp', 'millisecond precision to fill the first 48 bits')),
]
norm = lambda s: re.sub(r'\s+', ' ', s).lower()
missing = [f for _, n, facts in qa for f in facts if norm(f) not in norm(CORPUS[n])]
leaks = [(f, d) for _, n, facts in qa for f in facts for d in CORPUS if d != n and norm(f) in norm(CORPUS[d])]
assert not missing and not leaks, (missing, leaks)
print(len(qa), 'queries; every answer string found only in its own RFC')

10 queries; every answer string found only in its own RFC


In [4]:
indices = {n: build(c) for n, c in configs.items()}

# Relevance labels are fixed from the chunk text alone, before any retrieval runs.
labels = {name: [{i for i, (n, t) in enumerate(chunks) if n == rfc and any(norm(f) in norm(t) for f in facts)}
                 for _, rfc, facts in qa]
          for name, chunks in configs.items()}

K = 3
retrieved = {name: [set(search(indices[name], q, K)) for q, _, _ in qa] for name in configs}

print(f'{"config":16s} {"P@3":>5s} {"R@3":>5s} {"hit@3":>6s} {"doc-P@3":>8s} {"split":>6s}')
for name, chunks in configs.items():
    rows = list(zip(retrieved[name], labels[name], qa))
    p = np.mean([len(ids & rel) / K for ids, rel, _ in rows])
    # No relevant chunk means chunking split every occurrence of the answer; score recall 0.
    r = np.mean([len(ids & rel) / len(rel) if rel else 0 for ids, rel, _ in rows])
    hit = np.mean([bool(ids & rel) for ids, rel, _ in rows])
    doc_p = np.mean([sum(chunks[i][0] == rfc for i in ids) / K for ids, _, (_, rfc, _) in rows])
    split = sum(not rel for rel in labels[name])
    print(f'{name:16s} {p:5.2f} {r:5.2f} {hit:6.2f} {doc_p:8.2f} {split:6d}')

# Per query: relevant chunks retrieved / relevant chunks that exist.
print(f'\n{"query":46s}' + ''.join(f'{n:>16s}' for n in configs))
for j, (q, _, _) in enumerate(qa):
    cells = [f'{len(retrieved[n][j] & labels[n][j])}/{len(labels[n][j])}' if labels[n][j] else 'split'
             for n in configs]
    print(f'{q[:45]:46s}' + ''.join(f'{c:>16s}' for c in cells))

config             P@3   R@3  hit@3  doc-P@3  split
small (120/20)    0.03  0.03   0.10     0.93      1
large (320/40)    0.20  0.42   0.60     0.97      0
by-paragraph      0.27  0.53   0.70     0.97      0

query                                           small (120/20)  large (320/40)    by-paragraph
what text encoding must JSON use when exchang              0/2             1/2             1/2
what character separates the date from the ti              0/3             0/3             0/3
which character replaces the plus sign in URL              0/1             0/1             0/1
which claim stops a JWT from being used befor            split             1/1             1/1
what status code should a server return for a              0/2             1/2             1/1
response when the token lacks the permissions              0/1             1/1             1/1
if a cookie sets both Max-Age and Expires, wh              0/1             1/1             1/1
which opcode is used for a webs

**Metrics** 
- **P@3**: the share of the 3 retrieved chunks that are relevant. When a query has only one relevant chunk, its best possible P@3 is 0.33
- **R@3**: the share of a query's relevant chunks that were retrieved. R@3 is capped below 1 even though one hit is enough to answer
- **hit@3**: whether at least one relevant chunk was retrieved. Yes/no can the generator answer
- **doc-P@3**: the share of retrieved chunks that come from the correct RFC.
- **split**: queries where chunking left no chunk with a complete answer. introduces recall:0 hits


The precision (P@3) and recall (R@3) numbers vary more for the larger chunking strategies because recall measures the relevant chunks existing in the corpus while precision scores the retrieved chunks. When information is grouped in larger clusters more chunks contain relevancy hits for any one query, causing higher recall scores. With smaller chunks the pattern is inverted and less chunks are found relevant for any one query.

## Part 3: Generate (needs a key)
Build a grounded prompt from the retrieved chunks and answer with Gemini.

In [5]:
def gemini_chat(messages, model='gemini-3.5-flash', **kw):
    if not os.environ.get('GEMINI_API_KEY'):
        return None
    from openai import OpenAI
    client = OpenAI(api_key=os.environ['GEMINI_API_KEY'], base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return client.chat.completions.create(model=model, messages=messages, **kw).choices[0].message.content

def answer(q, config='by-paragraph'):
    ctx = retrieve(indices[config], configs[config], q, k=3)
    prompt = 'Answer only from the context.\nContext:\n- ' + '\n- '.join(f'[RFC {n}] {t}' for n, t in ctx) + f'\nQuestion: {q}\nAnswer:'
    out = gemini_chat([{'role':'user','content':prompt}])
    return out if out is not None else '[API-BLOCKED] set GEMINI_API_KEY to generate; grounded prompt was built'

print(answer('what status code should a server return for an expired access token'))

Based on the provided context, the server should respond with the **HTTP 401 (Unauthorized)** status code.


## Part 4 Failure

Section or paragraph based chunking splits headings into their own chunks which score high with relevant queries despite the fact that they contain no information themselves. The relevant information is also deprived of it's topical labeling isolated in it's own chunk. One mitigation is to include the concept of labels in the structure of the chunks by stripping headings and adding them to their associated chunks as an attribute, instead of allowing them to be chunked as normal text.



In [6]:
para = configs['by-paragraph']
for j in (1, 8):
    print('query:', qa[j][0])
    for i in search(indices['by-paragraph'], qa[j][0], K):
        print('  -', para[i][1][:110])

query: what character separates the date from the time in an internet timestamp
  - Date and Time on the Internet: Timestamps
  - There are many ways in which date and time values might appear in Internet protocols: this document focuses on
  - [RFC3339] Klyne, G. and C. Newman, "Date and Time on the Internet: Timestamps", RFC 3339, DOI 10.17487/RFC3339
query: what protocol version does the client send in the websocket opening handshake
  - The opening handshake is intended to be compatible with HTTP-based server-side software and intermediaries, so
  - 12. Using the WebSocket Protocol from Other Specifications
  - 11.3.4. Sec-WebSocket-Protocol


In [7]:
def relevant(chunks, rfc, facts):
    return {i for i, (n, t) in enumerate(chunks) if n == rfc and any(norm(f) in norm(t) for f in facts)}
def hits(chunks, idx):
    return [bool(set(search(idx, q, K)) & relevant(chunks, r, f)) for q, r, f in qa]

In [8]:
# Heading aware chunks
HEADING = re.compile(r'^(?:\d+(?:\.\d+)*\.?|Appendix [A-Z]\.?)\s+\S.{0,70}$')
TOC = re.compile(r'(?:\. ?){4,}')  # dot leaders: '....' or '. . . .'
def chunk_sections(text):
    out, heading = [], ''
    for p in chunk_paragraph(text):
        if TOC.search(p):
            continue
        if HEADING.match(p):
            heading = p; continue
        out.append(f'{heading}: {p}' if heading else p)
    return out

sections = chunk_corpus(chunk_sections)
sections_idx = build(sections)
para_hits, sec_hits = hits(para, indices['by-paragraph']), hits(sections, sections_idx)
print(f'by-paragraph {len(para)} chunks, hit@3 {sum(para_hits)}/{len(qa)}')
print(f'by-section   {len(sections)} chunks, hit@3 {sum(sec_hits)}/{len(qa)}\n')
print(f'{"query":46s} {"paragraph":>9s} {"section":>9s}')
for (q, _, _), a, b in zip(qa, para_hits, sec_hits):
    print(f'{q[:45]:46s} {a!s:>9s} {b!s:>9s}')

print('\nby-section, ping query retrieves:')
for i in search(sections_idx, qa[7][0], K):
    print('  -', sections[i][1][:110])

by-paragraph 2604 chunks, hit@3 7/10
by-section   2228 chunks, hit@3 7/10

query                                          paragraph   section
what text encoding must JSON use when exchang       True      True
what character separates the date from the ti      False      True
which character replaces the plus sign in URL      False     False
which claim stops a JWT from being used befor       True      True
what status code should a server return for a       True      True
response when the token lacks the permissions       True      True
if a cookie sets both Max-Age and Expires, wh       True      True
which opcode is used for a websocket ping fra       True     False
what protocol version does the client send in      False     False
how many bits of a UUIDv7 hold the millisecon       True      True

by-section, ping query retrieves:
  - 11.8. WebSocket Opcode Registry: Opcode The opcode denotes the frame type of the WebSocket frame, as defined i
  - 11.8. WebSocket Opcode Registry: R